# Collect Reasoning Chains
Collect chains of LLMs doing distractor generation on single math problems:
- simple prompt with non-reasoning model
- chain of thought with non-reasoning model
- simple prompt with reasoning model
- ls-informed prompt with reasoning model

**Paper mapping.** Produces the raw reasoning traces analyzed throughout the paper. Prompting settings -> Section 4 ("Models and prompting") and Appendix A.3; learning-science-informed prompt -> Appendix C.3; `*-naive-correct-*` runs (correct answer revealed) -> the correct-solution ablation in Section 4.3; the extra reasoner models (gpt-oss, gpt-5, Gemini, GLM-4.7-flash, Gemma-3) -> the model-diversity breadth check in Appendix C.3 (Table 24).

In [ ]:
import os
import json
from tqdm import tqdm
from collections import defaultdict
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI

from src.datasets import ADataset, get_or_create_dataset
from src.equality import MathSemanticEqualityChecker, ScienceSemanticEqualityChecker
from src.models.joint import JointModel
from src.models.impl.naive import DeepseekNaiveJointModel, OpenRouterNaiveJointModel, OpenAINaiveJointModel, VLLMNaiveJointModel
from src.models.impl.enforce_process import DeepseekEnforceProcessJointModel, OpenRouterEnforceProcessJointModel, OpenAIEnforceProcessJointModel, VLLMEnforceProcessJointModel
from src.models.impl.naive_cot import DeepseekNaiveCoTJointModel, OpenRouterNaiveCoTJointModel
from src.models.impl.naive_n_correct import DeepseekNaiveCorrectJointModel
from src.model_configurations import (
    gpt_4_1_mini_det_config, deepseek_chat, openrouter_glm_4_7_chat,
    deepseek_reasoner, openrouter_glm_4_7_reason, openrouter_gpt_oss_20b_reason,
    openrouter_gpt_oss_120b_reason, openrouter_glm_4_7_flash_reason,
    gpt_4_1, gpt_5,
    gemini_2_5_pro, gemini_2_5_flash,
    vllm_gemma_4_31b,
)

load_dotenv()

openai_client = OpenAI()

# Gemini exposes an OpenAI-compatible endpoint, so we reuse OpenAINaive*/OpenAIEnforce* with a
# dedicated client pointed at Google's base_url.
gemini_client = OpenAI(
    base_url=gemini_2_5_pro["base_url"],
    api_key=os.getenv(gemini_2_5_pro["api_key_var"]),
)

# Self-hosted vLLM server exposing an OpenAI-compatible endpoint (see vllm_gemma_4_31b's base_url).
vllm_client = OpenAI(base_url=vllm_gemma_4_31b["base_url"], api_key="EMPTY")

In [ ]:
def run_joint(data_folder: str, dataset: ADataset, setting_name: str, model: JointModel, num_distractors: int, save_every_n_questions: int = 4):
    import logging
    
    results_folder = os.path.join(data_folder, "joint_results")
    os.makedirs(results_folder, exist_ok=True)
    response_file = os.path.join(results_folder, f"{setting_name}_responses_by_datapointid.json")

    responses_by_datapointid = defaultdict(dict)
    if os.path.exists(response_file): 
        with open(response_file, "r+") as f:
            responses_by_datapointid = json.load(f)

    logger = logging.getLogger(setting_name)
    failed_datapoints = []

    # Process a single datapoint: try once without streaming, fall back to streaming
    def process_datapoint(i):
        datapoint_id = str(i)
        if datapoint_id in responses_by_datapointid:
            return None  # Already processed

        context = dataset[i]

        # First attempt: fast non-streaming call
        try:
            _, distractors_meta = model.generate_distractors(context, num_distractors=num_distractors)
            return (datapoint_id, distractors_meta)
        except Exception as first_error:
            # Fallback: streaming so partial output is captured if the model hits its token limit
            try:
                _, distractors_meta = model.generate_distractors(context, num_distractors=num_distractors, stream=True)
                return (datapoint_id, distractors_meta)
            except TypeError:
                # Model doesn't support stream kwarg — report the original error
                error_msg = f"Failed to process datapoint {datapoint_id}: {str(first_error)}"
                logger.error(error_msg)
                failed_datapoints.append((datapoint_id, str(first_error)))
                return None
            except Exception as e:
                error_msg = f"Failed to process datapoint {datapoint_id} (streaming fallback): {str(e)}"
                logger.error(error_msg)
                failed_datapoints.append((datapoint_id, str(e)))
                return None

    # Collect unprocessed indices
    unprocessed_indices = [i for i in range(len(dataset)) if str(i) not in responses_by_datapointid]
    
    if not unprocessed_indices:
        print(f"All datapoints already processed for {setting_name}")
        return

    # Process in parallel with checkpointing
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(process_datapoint, i): i for i in unprocessed_indices}
        
        completed_count = 0
        for future in tqdm(as_completed(futures), total=len(futures)):
            result = future.result()
            if result is not None:
                datapoint_id, distractors_meta = result
                responses_by_datapointid[datapoint_id] = distractors_meta
                completed_count += 1
                
                if completed_count % save_every_n_questions == 0:
                    with open(response_file, "w+") as f:
                        json.dump(responses_by_datapointid, f)

    # Final save
    with open(response_file, "w+") as f:
        json.dump(responses_by_datapointid, f)

    if failed_datapoints:
        print(f"\n{setting_name}: {len(failed_datapoints)} datapoints failed:")
        for datapoint_id, error in failed_datapoints[:10]:
            print(f"  - Datapoint {datapoint_id}: {error}")
        if len(failed_datapoints) > 10:
            print(f"  ... and {len(failed_datapoints) - 10} more")


## EEDI

In [ ]:
# Pilot on a true subset of the 500 by setting EEDI_N_LIMIT < 500.
# `get_or_create_dataset` derives dataset-{N}.json as the first-N prefix of dataset-500.json,
# so dataset indices align: scaling 50 -> 500 reuses already-collected joint_results entries.
EEDI_N_LIMIT = 500

data_folder = "eedi_data"
dataset = get_or_create_dataset(data_folder, n_limit=EEDI_N_LIMIT)

### Generate Reasoning Traces

#### Direct

In [ ]:
# -> Table 2, Direct row (Eedi, DS / GLM)
run_joint(data_folder, dataset, f"deepseek-naive-deepseek-chat",
            model=DeepseekNaiveJointModel(deepseek_chat, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-naive-z-ai_glm-4.7-chat",
            model=OpenRouterNaiveJointModel(openrouter_glm_4_7_chat, subject="math"), num_distractors=3)

#### CoT

In [ ]:
# -> Table 2, CoT row (Eedi, DS / GLM)
run_joint(data_folder, dataset, f"deepseek-naive-cot-deepseek-chat",
          model=DeepseekNaiveCoTJointModel(deepseek_chat, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-naive-cot-z-ai_glm-4.7-chat", 
          model=OpenRouterNaiveCoTJointModel(openrouter_glm_4_7_chat, subject="math"), num_distractors=3)

#### Reasoning

In [ ]:
# -> Table 2, Reasoning row (Eedi, DS) — this is also the DS model annotated in the main analysis
run_joint(data_folder, dataset, f"deepseek-naive-deepseek-reasoner",
            model=DeepseekNaiveJointModel(deepseek_reasoner, subject="math"), num_distractors=3)

# -> Section 4.3 correct-solution ablation (Eedi: 0.52 -> 0.56)
run_joint(data_folder, dataset, f"deepseek-naive-correct-deepseek-reasoner",
            model=DeepseekNaiveCorrectJointModel(deepseek_reasoner, subject="math"), num_distractors=3)

# -> Table 2, Reasoning row (Eedi, GLM) — also the GLM model annotated in the main analysis
run_joint(data_folder, dataset, f"openrouter-naive-z-ai_glm-4.7-reasoner",
            model=OpenRouterNaiveJointModel(openrouter_glm_4_7_reason, subject="math"), num_distractors=3)

# Below: extra reasoner models, Eedi-only, for the model-diversity breadth check (Appendix C.3, Table 24, w/o LS column)
run_joint(data_folder, dataset, f"openrouter-naive-z-ai_glm-4.7-flash-reasoner",
            model=OpenRouterNaiveJointModel(openrouter_glm_4_7_flash_reason, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-naive-openai_gpt-oss-20b-reasoner",
            model=OpenRouterNaiveJointModel(openrouter_gpt_oss_20b_reason, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-naive-openai_gpt-oss-120b-reasoner",
            model=OpenRouterNaiveJointModel(openrouter_gpt_oss_120b_reason, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openai-naive-gpt-5",
            model=OpenAINaiveJointModel(openai_client, gpt_5, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"gemini-naive-gemini-2.5-pro",
            model=OpenAINaiveJointModel(gemini_client, gemini_2_5_pro, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"gemini-naive-gemini-2.5-flash",
            model=OpenAINaiveJointModel(gemini_client, gemini_2_5_flash, subject="math"), num_distractors=3)

In [ ]:
# Gemma-3 (31b), Eedi — Appendix C.3, Table 24, w/o LS column
run_joint(data_folder, dataset, "vllm-naive-google_gemma-4-31b-it",
            model=VLLMNaiveJointModel(vllm_client, vllm_gemma_4_31b, subject="math"), num_distractors=3)

#### Learning-Science Informed

In [ ]:
# -> Appendix C.3: DS/GLM w/ LS column of Table 24 (no significant change vs. neutral prompt)
run_joint(data_folder, dataset, f"deepseek-enforce-process-deepseek-reasoner", 
            model=DeepseekEnforceProcessJointModel(deepseek_reasoner, show_correct=False, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-enforce-process-z-ai_glm-4.7-reasoner", 
            model=OpenRouterEnforceProcessJointModel(openrouter_glm_4_7_reason, show_correct=False, subject="math"), num_distractors=3)

# Below: extra reasoner models w/ LS, model-diversity breadth check (Appendix C.3, Table 24, w/ LS column)
run_joint(data_folder, dataset, f"openrouter-enforce-process-z-ai_glm-4.7-flash-reasoner", 
            model=OpenRouterEnforceProcessJointModel(openrouter_glm_4_7_flash_reason, show_correct=False, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-enforce-process-openai_gpt-oss-20b-reasoner", 
            model=OpenRouterEnforceProcessJointModel(openrouter_gpt_oss_20b_reason, show_correct=False, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openrouter-enforce-process-openai_gpt-oss-120b-reasoner", 
            model=OpenRouterEnforceProcessJointModel(openrouter_gpt_oss_120b_reason, show_correct=False, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"openai-enforce-process-gpt-5",
            model=OpenAIEnforceProcessJointModel(openai_client, gpt_5, show_correct=False, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"gemini-enforce-process-gemini-2.5-pro",
            model=OpenAIEnforceProcessJointModel(gemini_client, gemini_2_5_pro, show_correct=False, subject="math"), num_distractors=3)

run_joint(data_folder, dataset, f"gemini-enforce-process-gemini-2.5-flash",
            model=OpenAIEnforceProcessJointModel(gemini_client, gemini_2_5_flash, show_correct=False, subject="math"), num_distractors=3)

In [ ]:
# Gemma-3 (31b) w/ LS, Eedi — Appendix C.3, Table 24, w/ LS column
run_joint(data_folder, dataset, "vllm-enforce-process-google_gemma-4-31b-it",
            model=VLLMEnforceProcessJointModel(vllm_client, vllm_gemma_4_31b, show_correct=False, subject="math"), num_distractors=3)

## SciQ

In [ ]:
sciq_data_folder = "sciq_data"
sciq_dataset = get_or_create_dataset(sciq_data_folder, n_limit=500)

### Generate Reasoning Traces

#### Direct

In [ ]:
# -> Table 2, Direct row (SciQ, DS / GLM)
run_joint(sciq_data_folder, sciq_dataset, "deepseek-naive-deepseek-chat",
          model=DeepseekNaiveJointModel(deepseek_chat, subject="science"), num_distractors=3)

run_joint(sciq_data_folder, sciq_dataset, "openrouter-naive-z-ai_glm-4.7-chat",
          model=OpenRouterNaiveJointModel(openrouter_glm_4_7_chat, subject="science"), num_distractors=3)

#### CoT

In [ ]:
# -> Table 2, CoT row (SciQ, DS / GLM)
run_joint(sciq_data_folder, sciq_dataset, "deepseek-naive-cot-deepseek-chat",
          model=DeepseekNaiveCoTJointModel(deepseek_chat, subject="science"), num_distractors=3)

run_joint(sciq_data_folder, sciq_dataset, "openrouter-naive-cot-z-ai_glm-4.7-chat",
          model=OpenRouterNaiveCoTJointModel(openrouter_glm_4_7_chat, subject="science"), num_distractors=3)

#### Reasoning

In [ ]:
# -> Table 2, Reasoning row (SciQ, DS / GLM) — the two models annotated in the main analysis
run_joint(sciq_data_folder, sciq_dataset, "deepseek-naive-deepseek-reasoner",
          model=DeepseekNaiveJointModel(deepseek_reasoner, subject="science"), num_distractors=3)

# -> Section 4.3 correct-solution ablation (SciQ: 0.14 -> 0.18, footnote 3)
run_joint(sciq_data_folder, sciq_dataset, "deepseek-naive-correct-deepseek-reasoner",
          model=DeepseekNaiveCorrectJointModel(deepseek_reasoner, subject="science"), num_distractors=3)

run_joint(sciq_data_folder, sciq_dataset, "openrouter-naive-z-ai_glm-4.7-reasoner",
          model=OpenRouterNaiveJointModel(openrouter_glm_4_7_reason, subject="science"), num_distractors=3)

# Below: extra reasoner models, SciQ generalization check (Appendix C.3: "0.11-0.15 band for every model we ran there")
run_joint(sciq_data_folder, sciq_dataset, "openrouter-naive-z-ai_glm-4.7-flash-reasoner",
          model=OpenRouterNaiveJointModel(openrouter_glm_4_7_flash_reason, subject="science"), num_distractors=3)

run_joint(sciq_data_folder, sciq_dataset, "openrouter-naive-openai_gpt-oss-20b-reasoner",
          model=OpenRouterNaiveJointModel(openrouter_gpt_oss_20b_reason, subject="science"), num_distractors=3)

run_joint(sciq_data_folder, sciq_dataset, "openrouter-naive-openai_gpt-oss-120b-reasoner",
          model=OpenRouterNaiveJointModel(openrouter_gpt_oss_120b_reason, subject="science"), num_distractors=3)